# VNFood Vision - Build FAISS Index
Notebook này được sử dụng trên Google Colab để trích xuất đặc trưng (Feature Extraction) từ hàng ngàn ảnh trong tập dữ liệu và xây dựng FAISS Index phục vụ truy xuất hình ảnh tương đồng (Image Retrieval).

In [ ]:
!pip install faiss-cpu
!pip install timm torch torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import glob
import numpy as np
import faiss
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

# Cấu hình đường dẫn tới thư mục ảnh trên Drive
# THAY ĐỔI ĐƯỜNG DẪN NÀY ĐÚNG VỚI FOLDER CỦA BẠN TRÊN DRIVE
DATA_DIR = '/content/drive/MyDrive/VietFood-Project/data/processed/train'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Khởi tạo mô hình EfficientNet-B3 (chỉ lấy features, bỏ layer classification)
model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
# Loại bỏ lớp Fully Connected cuối cùng để lấy Feature Vector
model.classifier = nn.Identity()
model = model.to(device)
model.eval()

# Chuẩn bị Transform
transform = transforms.Compose([
    transforms.Resize(332),
    transforms.CenterCrop(300),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Lấy danh sách toàn bộ ảnh
image_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png'):
    image_paths.extend(glob.glob(os.path.join(DATA_DIR, '**', ext), recursive=True))

print(f"Found {len(image_paths)} images to process.")

# Lưu trữ Embedding (Features) và Metadata (Paths)
embeddings = []
valid_paths = []

batch_size = 32
for i in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[i:i+batch_size]
    batch_tensors = []
    
    for path in batch_paths:
        try:
            img = Image.open(path).convert('RGB')
            tensor = transform(img)
            batch_tensors.append(tensor)
            valid_paths.append(path)
        except Exception as e:
            print(f"Skipping {path}: {e}")
            
    if not batch_tensors: continue
    
    inputs = torch.stack(batch_tensors).to(device)
    with torch.no_grad():
        features = model(inputs)
        embeddings.append(features.cpu().numpy())

if embeddings:
    all_embeddings = np.vstack(embeddings)
    print(f"Extract complete. Shape: {all_embeddings.shape}")

In [ ]:
# Xây dựng FAISS Index (L2 distance)
import pickle

d = all_embeddings.shape[1] # Số chiều của vector (VD: 1536)
index = faiss.IndexFlatL2(d)
index.add(all_embeddings.astype(np.float32))

# Lưu Index và Metadata (Paths) để tái sử dụng trên Backend
faiss.write_index(index, 'faiss_index.bin')

metadata = {"paths": valid_paths}
with open('faiss_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
    
print("Saved faiss_index.bin and faiss_metadata.pkl")
print("Vui lòng tải 2 file này về máy tính và đặt vào thư mục 'backend/data/' của đồ án.")